# 29 — Integration Constraints

**Version:** `29_integration_constraints_v1`

Current repo progression:

\[
00:\ \text{Why multiplexing?}
\]

\[
07:\ \text{How many modes fit?}
\]

\[
13:\ \text{Which modes pair?}
\]

\[
23:\ \text{How do pairs become networks?}
\]

Notebook 29 asks:

> **What limits the transition from a multipartite graph to an integrated quantum device?**

The guiding idea:

\[
\text{multiplexing creates abundance}
\]

\[
\text{integration creates constraints}
\]

In [ ]:
from pathlib import Path
import json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import FancyBboxPatch

VERSION = "29_integration_constraints_v1"
print("running:", VERSION)

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    ROOT = cwd.parent
elif (cwd / "notebooks").exists():
    ROOT = cwd
else:
    ROOT = cwd

FIGURES_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
CSV_DIR = RESULTS_DIR / "csv"
JSON_DIR = RESULTS_DIR / "json"

for path in [FIGURES_DIR, CSV_DIR, JSON_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)

## 1. Resource stack

The seminar logic can be represented as a resource stack:

\[
\text{pump}
\rightarrow
\text{microresonator}
\rightarrow
\text{frequency comb}
\rightarrow
\text{mode pairs}
\rightarrow
\text{multipartite graph}
\rightarrow
\text{integrated quantum device}
\]

In [ ]:
stack = [
    "Pump",
    "Microresonator",
    "Frequency comb",
    "Mode pairs",
    "Multipartite graph",
    "Integrated quantum device",
]

fig, ax = plt.subplots(figsize=(8, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, len(stack) + 1)
ax.axis("off")

for i, label in enumerate(stack):
    y = len(stack) - i
    box = FancyBboxPatch(
        (0.25, y - 0.25),
        0.5,
        0.42,
        boxstyle="round,pad=0.03,rounding_size=0.03",
        linewidth=1.5,
        facecolor="white",
        edgecolor="black",
    )
    ax.add_patch(box)
    ax.text(0.5, y - 0.04, label, ha="center", va="center", fontsize=11)

    if i < len(stack) - 1:
        ax.annotate(
            "",
            xy=(0.5, y - 0.62),
            xytext=(0.5, y - 0.28),
            arrowprops=dict(arrowstyle="->", linewidth=1.8),
        )

ax.set_title("Resource Stack: From Comb to Integrated Device", fontsize=14, fontweight="bold")

fig.tight_layout()
resource_stack_path = FIGURES_DIR / "29_resource_stack.png"
fig.savefig(resource_stack_path, dpi=200)
plt.show()

print("saved:", resource_stack_path)

## 2. Scaling bottlenecks

A useful integration view is layered.

Each layer introduces a different resource or constraint.

In [ ]:
bottlenecks = pd.DataFrame([
    {
        "layer": "comb generation",
        "resource": "Q factor",
        "constraint": "loss and linewidth set usable comb quality"
    },
    {
        "layer": "multiplexing",
        "resource": "mode spacing",
        "constraint": "spacing controls channel density and addressability"
    },
    {
        "layer": "pair creation",
        "resource": "pump power",
        "constraint": "nonlinear generation depends on drive and resonator properties"
    },
    {
        "layer": "network formation",
        "resource": "connectivity",
        "constraint": "isolated pairs require additional coupling or measurement structure"
    },
    {
        "layer": "chip integration",
        "resource": "routing",
        "constraint": "frequency modes must be filtered, routed, and manipulated"
    },
    {
        "layer": "measurement",
        "resource": "detectors",
        "constraint": "large graphs create readout and detector burden"
    },
])

bottlenecks_path = CSV_DIR / "29_scaling_bottlenecks.csv"
bottlenecks.to_csv(bottlenecks_path, index=False)

bottlenecks

## 3. Constraint graph

The constraint graph treats implementation as a dependency structure.

This is not a physical simulation.

It asks which variables feed into which burdens.

In [ ]:
constraint_edges = [
    ("Q factor", "usable bandwidth"),
    ("mode spacing", "mode count"),
    ("usable bandwidth", "mode count"),
    ("mode count", "pair count"),
    ("pair count", "network size"),
    ("connectivity", "network size"),
    ("network size", "routing burden"),
    ("network size", "detector burden"),
    ("routing burden", "integration difficulty"),
    ("detector burden", "integration difficulty"),
]

G_constraints = nx.DiGraph()
G_constraints.add_edges_from(constraint_edges)

pos = {
    "Q factor": (0, 4),
    "usable bandwidth": (1.3, 4),
    "mode spacing": (0, 3),
    "mode count": (1.3, 3.4),
    "pair count": (2.6, 3.4),
    "connectivity": (2.6, 2.2),
    "network size": (3.9, 3.0),
    "routing burden": (5.2, 3.6),
    "detector burden": (5.2, 2.4),
    "integration difficulty": (6.6, 3.0),
}

fig, ax = plt.subplots(figsize=(11, 6))
nx.draw_networkx_nodes(G_constraints, pos, node_size=1800, ax=ax)
nx.draw_networkx_edges(G_constraints, pos, arrows=True, arrowstyle="->", width=1.8, ax=ax)
nx.draw_networkx_labels(G_constraints, pos, font_size=9, ax=ax)

ax.set_title("Constraint Graph for Integrated Microcomb Scaling")
ax.axis("off")

fig.tight_layout()
constraint_graph_path = FIGURES_DIR / "29_constraint_graph.png"
fig.savefig(constraint_graph_path, dpi=200)
plt.show()

print("saved:", constraint_graph_path)

## 4. Detector scaling

Mode abundance can shift the bottleneck.

The limiting resource may no longer be mode generation.

It may become measurement.

We compare three schematic readout models:

- full readout
- sampled readout
- multiplexed readout

In [ ]:
network_sizes = np.array([10, 50, 100, 500, 1000])

detector_scaling = pd.DataFrame({
    "network_size": network_sizes,
    "full_readout_detectors": network_sizes,
    "sampled_readout_detectors": np.ceil(np.sqrt(network_sizes)).astype(int),
    "multiplexed_readout_detectors": np.ceil(np.log2(network_sizes)).astype(int),
})

detector_scaling_path = CSV_DIR / "29_detector_scaling.csv"
detector_scaling.to_csv(detector_scaling_path, index=False)

detector_scaling

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(detector_scaling["network_size"], detector_scaling["full_readout_detectors"], marker="o", label="full readout")
ax.plot(detector_scaling["network_size"], detector_scaling["sampled_readout_detectors"], marker="s", label="sampled readout")
ax.plot(detector_scaling["network_size"], detector_scaling["multiplexed_readout_detectors"], marker="^", label="multiplexed readout")

ax.set_title("Detector Scaling Burden")
ax.set_xlabel("Network size")
ax.set_ylabel("Detector count")
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
detector_scaling_figure_path = FIGURES_DIR / "29_detector_scaling.png"
fig.savefig(detector_scaling_figure_path, dpi=200)
plt.show()

print("saved:", detector_scaling_figure_path)

## 5. Routing burden

Routing can also become a bottleneck.

A simple schematic proxy is to compare how routing effort might grow with the number of independent pair channels.

This model is intentionally coarse. It is a systems prompt, not a device simulation.

In [ ]:
pair_counts = np.array([1, 10, 100, 1000])

routing = pd.DataFrame({
    "pair_count": pair_counts,
    "linear_routing_proxy": pair_counts,
    "mesh_like_routing_proxy": pair_counts * np.log2(pair_counts + 1),
})

routing_path = CSV_DIR / "29_routing_burden.csv"
routing.to_csv(routing_path, index=False)

routing

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(routing["pair_count"], routing["linear_routing_proxy"], marker="o", label="linear routing proxy")
ax.plot(routing["pair_count"], routing["mesh_like_routing_proxy"], marker="s", label="mesh-like routing proxy")

ax.set_title("Routing Burden Proxy")
ax.set_xlabel("Pair count")
ax.set_ylabel("Routing burden")
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
routing_figure_path = FIGURES_DIR / "29_routing_burden.png"
fig.savefig(routing_figure_path, dpi=200)
plt.show()

print("saved:", routing_figure_path)

## 6. Constraint cascade

The key architectural message is a cascade:

\[
\text{higher Q}
\rightarrow
\text{more bandwidth}
\rightarrow
\text{more modes}
\rightarrow
\text{more pairs}
\rightarrow
\text{larger graph}
\rightarrow
\text{harder routing and detection}
\]

In [ ]:
cascade = [
    "Higher Q",
    "More usable bandwidth",
    "More frequency modes",
    "More symmetric pairs",
    "Larger graph",
    "Routing burden",
    "Detector burden",
]

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, len(cascade) + 1)
ax.axis("off")

for i, label in enumerate(cascade):
    y = len(cascade) - i
    box = FancyBboxPatch(
        (0.23, y - 0.25),
        0.54,
        0.42,
        boxstyle="round,pad=0.03,rounding_size=0.03",
        linewidth=1.5,
        facecolor="white",
        edgecolor="black",
    )
    ax.add_patch(box)
    ax.text(0.5, y - 0.04, label, ha="center", va="center", fontsize=11)

    if i < len(cascade) - 1:
        ax.annotate(
            "",
            xy=(0.5, y - 0.62),
            xytext=(0.5, y - 0.28),
            arrowprops=dict(arrowstyle="->", linewidth=1.8),
        )

ax.set_title("Constraint Cascade: Abundance Creates Integration Burden", fontsize=14, fontweight="bold")

fig.tight_layout()
cascade_path = FIGURES_DIR / "29_constraint_cascade.png"
fig.savefig(cascade_path, dpi=200)
plt.show()

print("saved:", cascade_path)

## 7. Summary

Multiplexing creates abundance.

Integration creates constraints.

The limiting resource may no longer be mode generation.

It may become:

- routing
- detection
- graph management
- integration complexity

In [ ]:
summary = {
    "notebook": "29_integration_constraints",
    "version": VERSION,
    "question": "What limits the transition from a multipartite graph to an integrated quantum device?",
    "claim": "Multiplexing creates abundance; integration creates constraints.",
    "outputs": [
        "figures/29_resource_stack.png",
        "figures/29_constraint_graph.png",
        "figures/29_detector_scaling.png",
        "figures/29_routing_burden.png",
        "figures/29_constraint_cascade.png",
        "results/csv/29_scaling_bottlenecks.csv",
        "results/csv/29_detector_scaling.csv",
        "results/csv/29_routing_burden.csv",
        "results/json/29_integration_constraints_summary.json",
        "results/29_integration_constraints_outputs.zip"
    ],
}

summary_path = JSON_DIR / "29_integration_constraints_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))

## 8. Download outputs

Run this cell to package all Notebook 29 outputs.

In Google Colab, it starts a browser download.

In local Jupyter, it prints the zip path.

In [ ]:
zip_path = RESULTS_DIR / "29_integration_constraints_outputs.zip"

files_to_zip = [
    FIGURES_DIR / "29_resource_stack.png",
    FIGURES_DIR / "29_constraint_graph.png",
    FIGURES_DIR / "29_detector_scaling.png",
    FIGURES_DIR / "29_routing_burden.png",
    FIGURES_DIR / "29_constraint_cascade.png",
    CSV_DIR / "29_scaling_bottlenecks.csv",
    CSV_DIR / "29_detector_scaling.csv",
    CSV_DIR / "29_routing_burden.csv",
    JSON_DIR / "29_integration_constraints_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print("download package ready:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Local Jupyter: download manually from")
    print(zip_path)

In [ ]:
outputs = [
    FIGURES_DIR / "29_resource_stack.png",
    FIGURES_DIR / "29_constraint_graph.png",
    FIGURES_DIR / "29_detector_scaling.png",
    FIGURES_DIR / "29_routing_burden.png",
    FIGURES_DIR / "29_constraint_cascade.png",
    CSV_DIR / "29_scaling_bottlenecks.csv",
    CSV_DIR / "29_detector_scaling.csv",
    CSV_DIR / "29_routing_burden.csv",
    JSON_DIR / "29_integration_constraints_summary.json",
    RESULTS_DIR / "29_integration_constraints_outputs.zip",
]

for output in outputs:
    print("exists:", output.exists(), "→", output.relative_to(ROOT) if output.exists() else output)

## Takeaway

The repo has now moved from:

\[
\text{modes}
\rightarrow
\text{pairs}
\rightarrow
\text{networks}
\rightarrow
\text{integration constraints}
\]

Notebook 29 adds the implementation question:

> **What becomes limiting after multiplexing succeeds?**

The answer is not only mode count.

It is also routing, detection, graph management, and integration complexity.